In [1]:
import sys
import time
import pandas as pd
from xtquant.xttrader import XtQuantTrader #创建交易对象使用
from xtquant.xttype import StockAccount #订阅账户信息使用
from xtquant import xtconstant #执行交易的时候需要引入
from datetime import datetime #时间戳改为日期时间格式的时候使用
# 回调类,处理账户状态
from xtquant.xttrader import XtQuantTraderCallback

class MyXtQuantTraderCallback(XtQuantTraderCallback):
    def on_disconnected(self):
        """
        连接断开
        :return:
        """
        print(datetime.datetime.now(),'连接断开回调')

    def on_stock_order(self, order):
        """
        委托回报推送
        :param order: XtOrder对象
        :return:
        """
        print(datetime.datetime.now(), '委托回调', order.order_remark)


    def on_stock_trade(self, trade):
        """
        成交变动推送
        :param trade: XtTrade对象
        :return:
        """
        print(datetime.datetime.now(), '成交回调', trade.order_remark)


    def on_order_error(self, order_error):
        """
        委托失败推送
        :param order_error:XtOrderError 对象
        :return:
        """
        # print("on order_error callback")
        # print(order_error.order_id, order_error.error_id, order_error.error_msg)
        print(f"委托报错回调 {order_error.order_remark} {order_error.error_msg}")

    def on_cancel_error(self, cancel_error):
        """
        撤单失败推送
        :param cancel_error: XtCancelError 对象
        :return:
        """
        print(datetime.datetime.now(), sys._getframe().f_code.co_name)

    def on_order_stock_async_response(self, response):
        """
        异步下单回报推送
        :param response: XtOrderResponse 对象
        :return:
        """
        print(f"异步委托回调 {response.order_remark}")

    def on_cancel_order_stock_async_response(self, response):
        """
        收到撤单回调信息
        :param response: XtCancelOrderResponse 对象
        :return:
        """
        print(datetime.datetime.now(), sys._getframe().f_code.co_name)

    def on_account_status(self, status):
        """
        账号状态信息变动推送
        :param response: XtAccountStatus 对象
        :return:
        """
        print(datetime.datetime.now(), sys._getframe().f_code.co_name)

    def on_stock_position(self, position):
        """
        持仓变动推送，根据：https://blog.csdn.net/liuyukuan/article/details/128754695
        :param position: XtPosition对象
        :return:
        """
        print("on position callback")
        print(position.stock_code, position.volume)

    def on_connected(self):
            """
            连接成功推送
            """
            pass

    def on_stock_asset(self,asset):
            """
            资金变动推送，根据：https://blog.csdn.net/liuyukuan/article/details/128754695
            :param asset: XtAsset对象
            :return:
            """
            print("资金变动推送on asset callback")
            print(asset.account_id,asset.cash,asset.total_asset)


#——————————————————————————————————————————————————————————————————————————————————————————————————————
#设置你的path='' 文件夹userdata_mini前面改为自己的QMT安装路径信息，acc=''引号内填入自己的账号
path = r'F:\trading\东北证券NET专业版\userdata_mini'
acct = "51318497"
# 1.创建交易对象API实例
session_id = int(time.time())
xt_trader = XtQuantTrader(path, session_id)
# 2.创建并注册回调实例
callback = MyXtQuantTraderCallback()
xt_trader.register_callback(callback)


# 3.xttrader连接miniQMT终端
xt_trader.start()
if xt_trader.connect() == 0:print('【软件终端连接成功！】')
else: print('【软件终端连接失败！】','\n 请运行并登录miniQMT.EXE终端。','\n path=改成你的QMT安装路径')           


#——————————————————————————————————————————————————————————————————————————————————————————————————————
# 4.订阅账户信息
ID = StockAccount(acct)
subscribe_result = xt_trader.subscribe(ID)
if subscribe_result == 0:print('【账户信息订阅成功！】')
else: 
    print('【账户信息订阅失败！】','\n 账户配置错误，检查账号是否正确。','\n acct=""内填加你的账号')
    sys.exit() #如果运行环境，账户都没配置好，后面的代码就不执行
#打印账户信息
asset = xt_trader.query_stock_asset(ID)
print('-'*18,'【{0}】'.format(asset.account_id),'-'*18) 
if asset:print(f"资产总额: {asset.total_asset}\n"  
                f"持仓市值：{asset.market_value}\n"
                f"可用资金：{asset.cash}\n"
                f"在途资金：{asset.frozen_cash}")



【软件终端连接成功！】
【账户信息订阅成功！】
------------------ 【51318497】 ------------------
资产总额: 32390.92
持仓市值：32268.0
可用资金：122.12
在途资金：0.0


In [ ]:
import time

# 记录开始时间
start_time = time.time()

# 批量异步下单：5笔订单，瞬间提交
seq_list = []  # 存所有请求序号
for i in range(5):
    print(f"提交第{i+1}单（异步）")
    # 异步下单：这行代码0.001秒就完成，只返回请求序号，不等待券商处理
    seq = xt_trader.order_stock_async(
        account=ID,
        stock_code=f"60000{i}.SH",
        order_type=xtconstant.STOCK_BUY,
        order_volume=100,
        price_type=xtconstant.LATEST_PRICE,
        price=10.5 + i,
        strategy_name="strategy1",
        order_remark=f"async_order_{i+1}"
    )
    seq_list.append(seq)
    print(f"第{i+1}单提交成功，请求序号：{seq}")

# 计算提交5单的耗时
submit_time = time.time() - start_time
print(f"异步批量提交5单总耗时：{submit_time:.3f}秒")

# 后续：券商后台同时处理这5单，1秒后通过回调返回所有订单号
# 这里模拟等待回调结果（实际是自动触发on_order_stock_async_response）
time.sleep(1)
print("5单全部处理完成，回调收到所有订单号")

提交第1单（异步）
第1单提交成功，请求序号：4
提交第2单（异步）
第2单提交成功，请求序号：5
提交第3单（异步）
第3单提交成功，请求序号：6
提交第4单（异步）
第4单提交成功，请求序号：7
提交第5单（异步）
第5单提交成功，请求序号：8
异步批量提交5单总耗时：0.000秒
异步委托回调 async_order_2
异步委托回调 async_order_3
异步委托回调 async_order_4
异步委托回调 async_order_1
异步委托回调 async_order_5
委托报错回调 async_order_1 限价买入 [SH600000] [COUNTER] [120141][证券交易未初始化]
[init_date=20260213,curr_date=20260221]

委托报错回调 async_order_5 限价买入 [SH600004] [COUNTER] [120141][证券交易未初始化]
[init_date=20260213,curr_date=20260221]

5单全部处理完成，回调收到所有订单号
